# Predictive Maintenance: Machine Failure Prediction

Predict machine failures from machine operating conditions using the AI4I 2020 predictive-maintenance dataset.

**Workflow**
1. Load and inspect the dataset.
2. Remove identifier and failure-mode leakage columns.
3. Split the data using stratification.
4. Compare Logistic Regression, Random Forest, and Histogram Gradient Boosting.
5. Tune models using stratified cross-validation and F1.
6. Evaluate the selected model once on the held-out test set.
7. Save the fitted pipeline as `artifacts/best_model.joblib`.

> **Dataset required:** Download `ai4i2020.csv` and place it in the `data/` directory. See the README for details.


## 1. Imports and configuration

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("data/ai4i2020.csv")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)


## 2. Load and inspect the data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download ai4i2020.csv and place it in the data/ directory."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
display(df.head())
display(df["Machine failure"].value_counts(normalize=True).rename("proportion"))


## 3. Prepare features

The five failure-mode indicator columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) are excluded because they describe failure mechanisms and would leak information about the target. Identifier columns are also removed.

In [ ]:
TARGET = "Machine failure"
LEAKY_COLUMNS = ["TWF", "HDF", "PWF", "OSF", "RNF"]
IDENTIFIER_COLUMNS = ["UDI", "Product ID"]

required_columns = [TARGET, *LEAKY_COLUMNS, *IDENTIFIER_COLUMNS]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {missing_columns}")

X = df.drop(columns=[TARGET, *LEAKY_COLUMNS, *IDENTIFIER_COLUMNS]).copy()
y = df[TARGET].copy()

# Remove units in square brackets and normalize feature names.
X.columns = (
    X.columns.str.replace(r"\\s*\\[.*\\]", "", regex=True)
    .str.strip()
    .str.replace(" ", "_")
)

numeric_features = [
    "Air_temperature",
    "Process_temperature",
    "Rotational_speed",
    "Torque",
    "Tool_wear",
]
categorical_features = ["Type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

print("Features:", X.columns.tolist())
print("Target distribution:")
display(y.value_counts(normalize=True).rename("proportion"))


## 4. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")


## 5. Compare baseline models with cross-validation

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": HistGradientBoostingClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

pipelines = {
    name: Pipeline([("preprocessor", preprocessor), ("classifier", model)])
    for name, model in models.items()
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_rows = []
for name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )
    cv_rows.append(
        {
            "Model": name,
            **{metric: scores[f"test_{metric}"].mean() for metric in scoring},
        }
    )

cv_results = pd.DataFrame(cv_rows).set_index("Model").sort_values("f1", ascending=False)
display(cv_results.round(4))
print(f"Best baseline model by mean CV F1: {cv_results['f1'].idxmax()}")


## 6. Hyperparameter tuning

In [ ]:
parameter_grids = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    "Random Forest": {
        "classifier__n_estimators": [200, 400],
        "classifier__max_depth": [None, 10, 20],
        "classifier__min_samples_leaf": [1, 3, 5],
    },
    "Gradient Boosting": {
        "classifier__learning_rate": [0.03, 0.1, 0.2],
        "classifier__max_depth": [None, 3, 6],
        "classifier__max_iter": [100, 300],
    },
}

tuned_models = {}
tuning_rows = []

for name, pipeline in pipelines.items():
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grids[name],
        cv=cv,
        scoring="f1",
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    tuned_models[name] = search
    tuning_rows.append(
        {
            "Model": name,
            "Best CV F1": search.best_score_,
            "Best parameters": search.best_params_,
        }
    )

tuning_results = pd.DataFrame(tuning_rows).set_index("Model").sort_values(
    "Best CV F1", ascending=False
)
display(tuning_results)
best_model_name = max(tuned_models, key=lambda name: tuned_models[name].best_score_)
best_model = tuned_models[best_model_name].best_estimator_
print(f"Selected model: {best_model_name}")


## 7. Final test-set evaluation

The test set is held out from model selection and hyperparameter tuning. Treat these metrics as the final estimate of performance.

In [ ]:
test_rows = []
for name, search in tuned_models.items():
    estimator = search.best_estimator_
    predictions = estimator.predict(X_test)
    probabilities = estimator.predict_proba(X_test)[:, 1]
    test_rows.append(
        {
            "Model": name,
            "Precision": precision_score(y_test, predictions, zero_division=0),
            "Recall": recall_score(y_test, predictions, zero_division=0),
            "F1": f1_score(y_test, predictions, zero_division=0),
            "ROC AUC": roc_auc_score(y_test, probabilities),
        }
    )

test_results = pd.DataFrame(test_rows).set_index("Model").sort_values("F1", ascending=False)
display(test_results.round(4))

final_predictions = best_model.predict(X_test)
print(f"Classification report — {best_model_name}")
print(
    classification_report(
        y_test,
        final_predictions,
        target_names=["No failure", "Failure"],
        zero_division=0,
    )
)

ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test,
    display_labels=["No failure", "Failure"],
    cmap="Blues",
)
plt.title(f"Confusion matrix — {best_model_name}")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Save the trained model

In [ ]:
model_path = ARTIFACT_DIR / "best_model.joblib"
joblib.dump(best_model, model_path)
print(f"Saved fitted pipeline to: {model_path}")


## 9. Example prediction

Run this after training and saving the model. The input columns must match the feature columns used during training.

In [ ]:
example_machine = pd.DataFrame(
    [
        {
            "Type": "M",
            "Air_temperature": 300.5,
            "Process_temperature": 310.2,
            "Rotational_speed": 1500,
            "Torque": 40.0,
            "Tool_wear": 100,
        }
    ]
)

loaded_model = joblib.load(ARTIFACT_DIR / "best_model.joblib")
failure_probability = loaded_model.predict_proba(example_machine)[:, 1][0]
prediction = loaded_model.predict(example_machine)[0]

print(f"Predicted class: {'Failure' if prediction == 1 else 'No failure'}")
print(f"Estimated failure probability: {failure_probability:.1%}")
